In [1]:
# PHASE 2 - NOTEBOOK E (lunit, SEED 42): adjacency pretraining CONTINUED FROM Lunit ViT-S/16 (+ SimCLR-style self control)
# Kaggle: GPU on, Internet on.  Re-running this cell resumes from the latest checkpoint.
# Output files to add to your checkpoints dataset (new version):
#   adjclv4-lunit_vit_small_neighbor_s42_best.pt   and   adjclv4-lunit_vit_small_self_s42_best.pt
!pip install -q timm scikit-image datasets transformers

"""
====================================================================
PART-II / PHASE 2 -- ADJACENCY-AWARE CONTRASTIVE PRETRAINING
Cross-Centre Generalization in Histopathology (Camelyon17-WILDS)

Idea: spatially adjacent patches of the same slide are positive pairs
(InfoNCE); everything else in the batch is a negative.  Same data index,
same patient-level train/val split, same ImageNet init and same checkpoint
format as the Part-I jigsaw/reconstruction scripts, so the existing
downstream (frozen linear probe, LOCO, probes, XAI) code runs unchanged.

Design choices that guard against the obvious shortcut:
  * Neighbours share the slide's stain, so a plain contrastive loss can be
    solved by "same stain => same pair".  Therefore
      (1) every batch is drawn from ONE centre (negatives share the stain
          distribution of the positives)                    [CENTER_PURE_BATCHES]
      (2) both views get strong stain (HED) + colour augmentation [HED_P]
      (3) only SPATIALLY CLOSE same-slide patches are masked out of the negatives
          (plausible false negatives, within NEG_MASK_RADIUS cells of either view).
          Far-away patches of the SAME slide stay as hard negatives, so the
          slide's stain/texture/blur identity cannot separate positives from
          negatives; each batch is built from a few slides (SLIDES_PER_BATCH)
          so that such negatives are plentiful.                [NEG_MASK_RADIUS]
      (v1 masked ALL same-slide negatives, which left only other-slide negatives
       and let the model win by recognising the slide -- fixed here.)
  * POS_MODE='self' is the SimCLR-style control (two augmentations of the
    SAME patch) -- it tells you whether *adjacency* helps or just
    contrastive learning in general.

Run:
    python adjacency_contrastive.py --check
    python adjacency_contrastive.py --smoke
    python adjacency_contrastive.py --backbone resnet18 --seed 42
    python adjacency_contrastive.py --backbone vit_small --pos-mode self --tag simclr
====================================================================
"""

import os
import sys
import math
import random
import pickle
import gc
import argparse
import warnings
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image as PILImage
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')


# ====================================================================
# CONFIG
# ====================================================================
class Config:
    HF_DATASET = "wltjr1007/Camelyon17-WILDS"
    CACHE_DIR = "/kaggle/working/cache"
    RESULTS_DIR = "/kaggle/working/results"
    CHECKPOINT_DIR = "/kaggle/working/checkpoints"
    LATTICE_CACHE = None      # set to an existing Part-I lattice pickle to skip the rebuild

    TAG = 'adjclv4'

    # --- positives / negatives ---
    POS_MODE = 'neighbor'     # 'neighbor' | 'self' (SimCLR control) | 'mixed' (50/50)
    NEIGHBOR_RADIUS = 1       # Chebyshev radius on the patch grid (1 = 8-neighbourhood)
    TEMPERATURE = 0.1
    CENTER_PURE_BATCHES = True
    NEG_MASK_RADIUS = 2       # mask same-slide negatives whose cell is within this Chebyshev distance (None = no mask)
    SLIDES_PER_BATCH = 3      # slides (of one centre) represented in each batch -> many same-slide hard negatives

    # --- v3 additions ---
    INIT_HUB = None           # timm hf-hub id: initialise the ViT-Small backbone from e.g. a pathology foundation model
    ADV_LAMBDA = 0.0          # > 0: centre-adversarial head (gradient reversal) on the pooled features
    ADV_HIDDEN = 256
    CENTERS_PER_BATCH = 1     # > 1: each batch mixes this many centres; negatives stay within the same centre
    HF_PHIKON = 'owkin/phikon'   # BACKBONE='phikon': HuggingFace ViT-B/16 (CLS token, as in Part I)
    GRAD_CKPT = True          # gradient checkpointing for the ViT-B (keeps 64 pairs/batch inside 16 GB)
    HED_P = 0.8               # probability of HED stain perturbation per view

    # --- data (same pool + split as the jigsaw runs) ---
    GRID_SIZE = 3             # only used to rebuild the lattice index identically
    VAL_PATIENT_FRAC = 0.25
    SPLIT_SEED = 42           # fixed => same validation patients for every training seed
    TRAIN_CENTERS = [0, 1, 3, 4]
    HELD_OUT_CENTER = 2

    # --- model ---
    BACKBONE = 'resnet18'     # 'resnet18' | 'vit_small' | 'swin_tiny'
    CNN_INPUT_SIZE = 96
    VIT_INPUT_SIZE = 224
    PRETRAINED = True
    FREEZE_STAGES = 2         # ResNet18 only; 2 matches the Part-I jigsaw run
    PROJ_HIDDEN = 512
    PROJ_DIM = 128

    # --- training (per-backbone overrides applied in apply_backbone_defaults) ---
    TRAIN_PAIRS_PER_EPOCH = 40000
    VAL_PAIRS = 4000
    BATCH_SIZE = 128          # pairs per batch => 2*BATCH_SIZE views, 2*(B-1) negatives
    VAL_BATCH_SIZE = 128
    LR = 3e-4
    BACKBONE_LR_MULT = 1.0
    WARMUP_EPOCHS = 3
    WEIGHT_DECAY = 0.05
    EPOCHS = 60
    GRAD_CLIP = 1.0
    EARLY_STOP_PATIENCE = 10
    NUM_WORKERS = 4
    SEED = 42
    USE_AMP = True
    RESUME = True

    # --- smoke ---
    SMOKE_EPOCHS = 2
    SMOKE_TRAIN = 1024
    SMOKE_VAL = 512


def apply_backbone_defaults(cfg):
    """Batch size / LR multipliers per backbone (contrastive needs a real batch, so no
    gradient accumulation: it would silently shrink the negative pool)."""
    if cfg.BACKBONE == 'resnet18':
        cfg.BATCH_SIZE, cfg.VAL_BATCH_SIZE, cfg.TRAIN_PAIRS_PER_EPOCH = 128, 128, 40000
        cfg.BACKBONE_LR_MULT, cfg.FREEZE_STAGES = 1.0, cfg.FREEZE_STAGES
    elif cfg.BACKBONE == 'vit_small':
        cfg.BATCH_SIZE, cfg.VAL_BATCH_SIZE, cfg.TRAIN_PAIRS_PER_EPOCH = 64, 64, 10000
        cfg.BACKBONE_LR_MULT, cfg.FREEZE_STAGES = 0.1, 0
    elif cfg.BACKBONE == 'swin_tiny':
        cfg.BATCH_SIZE, cfg.VAL_BATCH_SIZE, cfg.TRAIN_PAIRS_PER_EPOCH = 32, 32, 8000
        cfg.BACKBONE_LR_MULT, cfg.FREEZE_STAGES = 0.1, 0
    elif cfg.BACKBONE == 'phikon':        # ViT-B/16: ~4x the cost of ViT-S, so fewer, shorter epochs
        cfg.BATCH_SIZE, cfg.VAL_BATCH_SIZE, cfg.TRAIN_PAIRS_PER_EPOCH = 64, 64, 4000
        cfg.BACKBONE_LR_MULT, cfg.FREEZE_STAGES = 0.1, 0
        cfg.EPOCHS, cfg.EARLY_STOP_PATIENCE = 30, 8
    else:
        raise ValueError(cfg.BACKBONE)


def get_input_size(backbone):
    return Config.VIT_INPUT_SIZE if backbone in ('vit_small', 'swin_tiny', 'phikon') else Config.CNN_INPUT_SIZE


def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def worker_init_fn(worker_id):
    seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(seed); random.seed(seed)


def make_dirs():
    for d in (Config.CACHE_DIR, Config.RESULTS_DIR, Config.CHECKPOINT_DIR):
        os.makedirs(d, exist_ok=True)


# ====================================================================
# DATA LOADING + SPATIAL INDEX  (identical to the Part-I scripts)
# ====================================================================
REQUIRED = ['patient', 'node', 'center', 'x_coord', 'y_coord']


def load_full_dataset():
    from datasets import load_dataset, concatenate_datasets
    ds = load_dataset(Config.HF_DATASET)
    parts = [ds[k] for k in ds.keys()]
    full = concatenate_datasets(parts) if len(parts) > 1 else parts[0]
    print(f"✓ {len(full):,} patches")
    return full


def check_schema(dataset):
    missing = [c for c in REQUIRED if c not in dataset.features]
    if missing:
        print(f"❌ missing columns: {missing}")
        return False
    print(f"✓ required columns present, {len(dataset):,} patches")
    return True


class Lattice:
    __slots__ = ('patient', 'node', 'center', 'idx_grid', 'valid_centers')

    def __init__(self, patient, node, center, patches, grid_size):
        self.patient, self.node, self.center = patient, node, center
        xs = sorted({p['x'] for p in patches})
        ys = sorted({p['y'] for p in patches})
        x_to_c = {x: i for i, x in enumerate(xs)}
        y_to_r = {y: i for i, y in enumerate(ys)}
        g = np.full((len(ys), len(xs)), -1, dtype=np.int64)
        for p in patches:
            g[y_to_r[p['y']], x_to_c[p['x']]] = p['idx']
        self.idx_grid = g
        half = grid_size // 2
        H, W = g.shape
        valid = []
        for r in range(half, H - half):
            for c in range(half, W - half):
                if (g[r - half:r + half + 1, c - half:c + half + 1] >= 0).all():
                    valid.append((r, c))
        self.valid_centers = np.array(valid, dtype=np.int32) if valid else np.zeros((0, 2), dtype=np.int32)


class SpatialIndex:
    def __init__(self, dataset, grid_size, cache_path=None):
        self.grid_size = grid_size
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                self.lattices = pickle.load(f)
            print(f"✓ loaded {len(self.lattices)} lattices from cache")
        else:
            self.lattices = self._build(dataset)
            if cache_path:
                with open(cache_path, 'wb') as f:
                    pickle.dump(self.lattices, f)

    def _build(self, dataset):
        meta = dataset.remove_columns(['image']).to_pandas()
        groups = defaultdict(list)
        for row in tqdm(meta.itertuples(index=True), total=len(meta), desc="index"):
            groups[(int(row.patient), int(row.node), int(row.center))].append(
                {'x': int(row.x_coord), 'y': int(row.y_coord), 'idx': int(row.Index)})
        out = []
        for (pat, node, ctr), patches in tqdm(groups.items(), desc="lattices"):
            lat = Lattice(pat, node, ctr, patches, self.grid_size)
            if len(lat.valid_centers):       # same pool as the jigsaw runs
                out.append(lat)
        return out

    def patients_in(self, centers):
        return sorted({l.patient for l in self.lattices if l.center in centers})

    def subset(self, centers, patients):
        pats = set(patients)
        return [l for l in self.lattices if l.center in centers and l.patient in pats]


# ====================================================================
# AUGMENTATION
# ====================================================================
def hed_stain_augment(img, alpha_range=(0.9, 1.1), beta_range=(-0.08, 0.08)):
    """Tellez et al. (2018) HED perturbation -- same settings as the Stain Augmentation baseline."""
    from skimage.color import rgb2hed, hed2rgb
    arr = np.asarray(img).astype(np.float32) / 255.0
    hed = rgb2hed(arr)
    for c in range(3):
        hed[..., c] = hed[..., c] * np.random.uniform(*alpha_range) + np.random.uniform(*beta_range)
    rgb = np.clip(hed2rgb(hed), 0, 1)
    return PILImage.fromarray((rgb * 255).astype(np.uint8))


class RandomRot90:
    def __call__(self, img):
        return img.rotate(90 * random.randrange(4))


class MaybeHED:
    def __init__(self, p):
        self.p = p
        try:
            import skimage  # noqa: F401
            self.ok = True
        except Exception:
            self.ok = False
            print("⚠ scikit-image missing: HED augmentation disabled (colour jitter only)")

    def __call__(self, img):
        if self.ok and self.p > 0 and random.random() < self.p:
            return hed_stain_augment(img)
        return img


NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])


def build_transforms(backbone, hed_p):
    size = get_input_size(backbone)
    train = transforms.Compose([
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), RandomRot90(),
        transforms.RandomResizedCrop(size, scale=(0.5, 1.0), ratio=(0.9, 1.1)),
        MaybeHED(hed_p),
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomGrayscale(p=0.15),
        transforms.ToTensor(), NORM])
    val = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(), NORM])
    return train, val


# ====================================================================
# DATASET  --  anchor + adjacent positive
# ====================================================================
class AdjacentPairDataset(Dataset):
    """Keys are (center, j).  Train: sampled on the fly (lattice ~ #cells in centre, random anchor,
    random occupied neighbour within the radius).  Val: deterministic given j."""

    def __init__(self, hf_dataset, lattices, transform, train, cfg):
        self.ds, self.transform, self.train, self.cfg = hf_dataset, transform, train, cfg
        self.lat_ids_by_center = defaultdict(list)
        self.lattices = lattices
        for li, lat in enumerate(lattices):
            self.lat_ids_by_center[lat.center].append(li)
        self.p_by_center, self.cells = {}, {}
        for c, ids in self.lat_ids_by_center.items():
            n = np.array([(lattices[i].idx_grid >= 0).sum() for i in ids], dtype=np.float64)
            self.p_by_center[c] = n / n.sum()
        R = cfg.NEIGHBOR_RADIUS
        self.offsets = [(dr, dc) for dr in range(-R, R + 1) for dc in range(-R, R + 1) if (dr, dc) != (0, 0)]

    def center_weights(self):
        cs = sorted(self.lat_ids_by_center)
        w = np.array([sum((self.lattices[i].idx_grid >= 0).sum() for i in self.lat_ids_by_center[c])
                      for c in cs], dtype=np.float64)
        return cs, w / w.sum()

    def _occupied(self, li):
        if li not in self.cells:
            self.cells[li] = np.argwhere(self.lattices[li].idx_grid >= 0)
        return self.cells[li]

    def _load(self, idx):
        img = self.ds[int(idx)]['image']
        return img.convert('RGB') if img.mode != 'RGB' else img

    def __len__(self):
        raise NotImplementedError("indexed through CenterBatchSampler keys")

    def __getitem__(self, key):
        center, j, li = key
        rs = np.random if self.train else np.random.RandomState(1000003 * (j + 1) + 17)
        rnd = random if self.train else random.Random(1000003 * (j + 1) + 17)
        lat = self.lattices[li]
        cells = self._occupied(li)
        r, c = cells[int(rs.randint(len(cells)))]

        mode = self.cfg.POS_MODE
        if mode == 'mixed':
            mode = 'neighbor' if rnd.random() < 0.5 else 'self'
        anchor = int(lat.idx_grid[r, c])
        pos, adjacent, pcell = anchor, False, (int(r), int(c))
        if mode == 'neighbor':
            H, W = lat.idx_grid.shape
            cand = [(int(lat.idx_grid[r + dr, c + dc]), (int(r + dr), int(c + dc))) for dr, dc in self.offsets
                    if 0 <= r + dr < H and 0 <= c + dc < W and lat.idx_grid[r + dr, c + dc] >= 0]
            if cand:                                  # isolated patch -> falls back to a 'self' pair
                (pos, pcell), adjacent = cand[rnd.randrange(len(cand))], True
        a, b = self._load(anchor), self._load(pos)
        if not self.train and not adjacent:     # self pair at validation: identical views would be trivial
            b = b.transpose(PILImage.FLIP_LEFT_RIGHT)
        return {'v1': self.transform(a), 'v2': self.transform(b),
                'lattice': li, 'center': center, 'adjacent': adjacent,
                'cell1': (int(r), int(c)), 'cell2': pcell}


class CenterBatchSampler:
    """Yields lists of (center, j, lattice_id) keys.  With CENTER_PURE_BATCHES every key in a batch has the
    same centre; each batch draws its pairs from `slides_per_batch` slides of that centre, so that
    same-slide (far-apart) patches are available as hard negatives."""

    def __init__(self, ds, n_pairs, batch_size, pure, slides_per_batch, seed=None, centers_per_batch=1):
        self.ds, self.pure, self.k, self.cpb = ds, pure, slides_per_batch, centers_per_batch
        self.centers, self.weights = ds.center_weights()
        self.n_batches = n_pairs // batch_size
        self.bs, self.seed = batch_size, seed
        self.epoch = 0

    def __len__(self):
        return self.n_batches

    def _slides(self, rng, c):
        ids, p = self.ds.lat_ids_by_center[c], self.ds.p_by_center[c]
        k = min(self.k, len(ids))
        return list(rng.choice(ids, size=k, replace=False, p=p))

    def __iter__(self):
        rng = np.random.RandomState(None if self.seed is None else self.seed + self.epoch)
        self.epoch += 1
        for b in range(self.n_batches):
            if self.pure and self.cpb > 1:
                k = min(self.cpb, len(self.centers))
                cs = [int(c) for c in rng.choice(self.centers, size=k, replace=False, p=self.weights)]
                per, batch = self.bs // k, []
                for ci, c in enumerate(cs):
                    n_c = per if ci < k - 1 else self.bs - per * (k - 1)
                    sl = self._slides(rng, c)
                    start = len(batch)
                    batch += [(c, b * self.bs + start + q, int(sl[q % len(sl)])) for q in range(n_c)]
                yield batch
            elif self.pure:
                c = int(rng.choice(self.centers, p=self.weights))
                sl = self._slides(rng, c)
                yield [(c, b * self.bs + k, int(sl[k % len(sl)])) for k in range(self.bs)]
            else:
                batch = []
                for k in range(self.bs):
                    c = int(rng.choice(self.centers, p=self.weights))
                    batch.append((c, b * self.bs + k, int(rng.choice(self.ds.lat_ids_by_center[c],
                                                                       p=self.ds.p_by_center[c]))))
                yield batch


def collate(batch):
    return {'v1': torch.stack([b['v1'] for b in batch]), 'v2': torch.stack([b['v2'] for b in batch]),
            'lattice': torch.tensor([b['lattice'] for b in batch]),
            'center': torch.tensor([b['center'] for b in batch]),
            'adjacent': torch.tensor([b['adjacent'] for b in batch]),
            'cells': torch.tensor([b['cell1'] for b in batch] + [b['cell2'] for b in batch])}


def create_dataloaders(hf_dataset, index, cfg):
    patients = index.patients_in(cfg.TRAIN_CENTERS)
    random.Random(cfg.SPLIT_SEED).shuffle(patients)          # same call as the jigsaw scripts
    n_val = max(2, int(round(cfg.VAL_PATIENT_FRAC * len(patients))))
    val_p, train_p = patients[:n_val], patients[n_val:]
    train_lat = index.subset(cfg.TRAIN_CENTERS, train_p)
    val_lat = index.subset(cfg.TRAIN_CENTERS, val_p)
    print(f"Patients: {len(train_p)} train / {len(val_p)} val | slides: {len(train_lat)} / {len(val_lat)}")
    ttf, vtf = build_transforms(cfg.BACKBONE, cfg.HED_P)
    train_ds = AdjacentPairDataset(hf_dataset, train_lat, ttf, True, cfg)
    val_ds = AdjacentPairDataset(hf_dataset, val_lat, vtf, False, cfg)
    tsamp = CenterBatchSampler(train_ds, cfg.TRAIN_PAIRS_PER_EPOCH, cfg.BATCH_SIZE, cfg.CENTER_PURE_BATCHES,
                               cfg.SLIDES_PER_BATCH, centers_per_batch=cfg.CENTERS_PER_BATCH)
    vsamp = CenterBatchSampler(val_ds, cfg.VAL_PAIRS, cfg.VAL_BATCH_SIZE, cfg.CENTER_PURE_BATCHES,
                               cfg.SLIDES_PER_BATCH, seed=cfg.SPLIT_SEED, centers_per_batch=cfg.CENTERS_PER_BATCH)
    # val batches must be identical every epoch -> freeze the sampler's output once
    vbatches = list(vsamp)
    common = dict(num_workers=cfg.NUM_WORKERS, pin_memory=True, worker_init_fn=worker_init_fn,
                  persistent_workers=cfg.NUM_WORKERS > 0, collate_fn=collate)
    return (DataLoader(train_ds, batch_sampler=tsamp, **common),
            DataLoader(val_ds, batch_sampler=vbatches, **common))


# ====================================================================
# MODEL
# ====================================================================
class PhikonBackbone(nn.Module):
    """owkin/phikon (HF ViT-B/16, iBOT) -> CLS token; identical to the Part-I HFTransformersViTWrapper pooling."""
    def __init__(self, hub='owkin/phikon', pretrained=True):
        super().__init__()
        from transformers import AutoModel, AutoConfig
        if pretrained:
            self.vit = AutoModel.from_pretrained(hub)
        else:
            self.vit = AutoModel.from_config(AutoConfig.from_pretrained(hub))
        self.dim = self.vit.config.hidden_size

    def forward(self, x):
        return self.vit(pixel_values=x).last_hidden_state[:, 0]


class _GradReverse(torch.autograd.Function):
    """Identity forward, gradient multiplied by -lambda backward (DANN gradient reversal)."""
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None


class AdjCLModel(nn.Module):
    """`backbone` / `pool` are built exactly like the Part-I ReconstructionModel, so a checkpoint's
    `model_state_dict` loads into the existing downstream ReconModel (strict=False ignores `head.*`)."""

    def __init__(self, name, pretrained=True, freeze_stages=0, hidden=512, out_dim=128,
                 init_hub=None, adv_classes=0, adv_hidden=256):
        super().__init__()
        self.backbone_name = name
        self.pool = None
        self._frozen_children = 0
        if name == 'resnet18':
            w = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
            net = models.resnet18(weights=w)
            self.feature_dim = 512
            self.backbone = nn.Sequential(*list(net.children())[:-1])
            self.pool = nn.AdaptiveAvgPool2d(1)
            if freeze_stages > 0:
                self._frozen_children = 4 + freeze_stages
                for m in list(self.backbone.children())[:self._frozen_children]:
                    for p in m.parameters():
                        p.requires_grad = False
        elif name == 'vit_small':
            import timm
            if init_hub:                                   # e.g. a pathology foundation model (same ViT-S/16 arch)
                self.backbone = timm.create_model(init_hub, pretrained=pretrained, num_classes=0)
            else:
                self.backbone = timm.create_model('vit_small_patch16_224', pretrained=pretrained,
                                                  num_classes=0, img_size=get_input_size(name))
            self.feature_dim = 384
        elif name == 'swin_tiny':
            import timm
            self.backbone = timm.create_model('swin_tiny_patch4_window7_224',
                                              pretrained=pretrained, num_classes=0)
            self.feature_dim = 768
        elif name == 'phikon':
            self.backbone = PhikonBackbone(init_hub or Config.HF_PHIKON, pretrained)
            self.feature_dim = self.backbone.dim
            if Config.GRAD_CKPT:
                self.backbone.vit.gradient_checkpointing_enable()
        else:
            raise ValueError(name)
        self.head = nn.Sequential(nn.Linear(self.feature_dim, hidden), nn.BatchNorm1d(hidden),
                                  nn.ReLU(inplace=True), nn.Linear(hidden, out_dim))
        self.adv_head = None
        self.grl_lambda = 0.0
        if adv_classes > 0:                                # centre classifier behind a gradient-reversal layer
            self.adv_head = nn.Sequential(nn.Linear(self.feature_dim, adv_hidden), nn.ReLU(inplace=True),
                                          nn.Linear(adv_hidden, adv_classes))

    def set_train(self):
        self.train()
        if self._frozen_children:                       # frozen stages stay in eval (BN stats fixed)
            for m in list(self.backbone.children())[:self._frozen_children]:
                m.eval()

    def encode(self, x):                                # == downstream feature (pooled backbone output)
        f = self.backbone(x)
        if f.dim() == 4:
            f = torch.flatten(self.pool(f), 1)
        elif f.dim() == 3:
            f = f.mean(1)
        return f

    def forward(self, x):
        return F.normalize(self.head(self.encode(x)).float(), dim=-1)

    def forward_all(self, x):
        f = self.encode(x)
        return F.normalize(self.head(f).float(), dim=-1), f

    def adv_logits(self, f):
        return self.adv_head(_GradReverse.apply(f.float(), self.grl_lambda))

    def param_groups(self, lr, mult):
        bb = [p for p in self.backbone.parameters() if p.requires_grad]
        hd = list(self.head.parameters())
        if self.adv_head is not None:
            hd += list(self.adv_head.parameters())
        return [{'params': bb, 'lr': lr * mult}, {'params': hd, 'lr': lr}]


# ====================================================================
# LOSS
# ====================================================================
def nt_xent(z1, z2, tau, lattice=None, cells=None, mask_radius=None, centers=None, same_center_only=False):
    """Symmetric InfoNCE over 2B views.  Negatives that lie on the SAME slide within `mask_radius` grid cells
    (Chebyshev) of the row's view are removed (plausible false negatives, incl. duplicates of the same patch);
    the genuine positive is always kept.  Returns loss, top-1 retrieval accuracy and the mean chance level
    (1 / number of remaining candidates per row)."""
    B = z1.size(0)
    z = torch.cat([z1, z2], 0).float()
    sim = z @ z.t() / tau
    eye = torch.eye(2 * B, dtype=torch.bool, device=z.device)
    target = torch.cat([torch.arange(B, 2 * B), torch.arange(0, B)]).to(z.device)
    mask = eye.clone()
    if mask_radius is not None and lattice is not None:
        l2 = torch.cat([lattice, lattice])
        same = l2[:, None] == l2[None, :]
        dist = (cells[:, None, :] - cells[None, :, :]).abs().max(-1).values
        is_pos = torch.zeros_like(eye)
        is_pos[torch.arange(2 * B), target] = True
        mask |= same & (dist <= mask_radius) & ~is_pos
    if same_center_only and centers is not None:       # mixed-centre batch: negatives only from the same centre
        c2 = torch.cat([centers, centers])
        mask |= (c2[:, None] != c2[None, :])
    sim = sim.masked_fill(mask, float('-inf'))
    loss = F.cross_entropy(sim, target)
    acc = (sim.argmax(1) == target).float().mean().item()
    chance = (1.0 / (~mask).sum(1).float()).mean().item()
    return loss, acc, chance


# ====================================================================
# TRAINER
# ====================================================================
def _scaler(enabled):
    try: return torch.amp.GradScaler('cuda', enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)


def _autocast(enabled):
    try: return torch.amp.autocast('cuda', enabled=enabled)
    except Exception: return torch.cuda.amp.autocast(enabled=enabled)


class Trainer:
    def __init__(self, model, device, cfg, steps_per_epoch):
        self.model, self.device, self.cfg = model, device, cfg
        self.opt = torch.optim.AdamW(model.param_groups(cfg.LR, cfg.BACKBONE_LR_MULT),
                                     weight_decay=cfg.WEIGHT_DECAY)
        total, warm = cfg.EPOCHS * steps_per_epoch, cfg.WARMUP_EPOCHS * steps_per_epoch
        self.total_steps, self.global_step = max(1, total), 0
        self.cidx = {c: i for i, c in enumerate(cfg.TRAIN_CENTERS)}
        self._nce, self._advacc = 0.0, None
        def lam(step):
            if step < warm:
                return step / max(1, warm)
            return 0.5 * (1 + math.cos(math.pi * min((step - warm) / max(1, total - warm), 1.0)))
        self.sched = torch.optim.lr_scheduler.LambdaLR(self.opt, lam)
        self.use_amp = cfg.USE_AMP and device.type == 'cuda'
        self.scaler = _scaler(self.use_amp)
        self.hist = defaultdict(list)
        self.start_epoch, self.best_val, self.bad = 0, float('inf'), 0
        self.stem = f"{cfg.TAG}_{cfg.BACKBONE}_{cfg.POS_MODE}_s{cfg.SEED}"

    def path(self, kind):
        return f"{self.cfg.CHECKPOINT_DIR}/{self.stem}_{kind}.pt"

    def save(self, epoch, best):
        ck = {'epoch': epoch, 'model_state_dict': self.model.state_dict(),
              'optimizer_state_dict': self.opt.state_dict(), 'scheduler_state_dict': self.sched.state_dict(),
              'scaler_state_dict': self.scaler.state_dict(), 'best_val': self.best_val, 'bad': self.bad,
              'history': dict(self.hist), 'backbone': self.cfg.BACKBONE, 'feature_dim': self.model.feature_dim,
              'cfg': {k: v for k, v in vars(Config).items() if k.isupper()}}
        torch.save(ck, self.path('latest'))
        if best:
            torch.save(ck, self.path('best'))

    def load(self, p):
        ck = torch.load(p, map_location=self.device)
        self.model.load_state_dict(ck['model_state_dict'])
        try:
            self.opt.load_state_dict(ck['optimizer_state_dict'])
            self.sched.load_state_dict(ck['scheduler_state_dict'])
            self.scaler.load_state_dict(ck['scaler_state_dict'])
        except Exception as e:
            print(f"⚠ optimizer state partial: {e}")
        self.start_epoch, self.best_val, self.bad = ck['epoch'] + 1, ck['best_val'], ck.get('bad', 0)
        self.hist = defaultdict(list, ck['history'])
        print(f"✓ resumed at epoch {self.start_epoch}, best val loss {self.best_val:.4f}")

    def _step_loss(self, batch):
        v1 = batch['v1'].to(self.device, non_blocking=True)
        v2 = batch['v2'].to(self.device, non_blocking=True)
        lat = batch['lattice'].to(self.device)
        cells = batch['cells'].to(self.device)
        ctr = batch['center'].to(self.device)
        x = torch.cat([v1, v2], 0)                     # one forward over both views: shared BatchNorm statistics
        adv = self.model.adv_head is not None
        if adv:
            z, f = self.model.forward_all(x)
        else:
            z = self.model(x)
        B = v1.size(0)
        nce, acc, ch = nt_xent(z[:B], z[B:], self.cfg.TEMPERATURE, lat, cells, self.cfg.NEG_MASK_RADIUS,
                               centers=ctr, same_center_only=self.cfg.CENTERS_PER_BATCH > 1)
        self._nce, self._advacc = nce.item(), None
        if not adv:
            return nce, acc, ch
        tgt = torch.tensor([self.cidx[int(c)] for c in torch.cat([ctr, ctr]).tolist()], device=self.device)
        logits = self.model.adv_logits(f)
        self._advacc = (logits.argmax(1) == tgt).float().mean().item()
        if self.model.training:                        # validation / checkpoint selection use the contrastive loss only
            return nce + F.cross_entropy(logits, tgt), acc, ch
        return nce, acc, ch

    def train_epoch(self, loader):
        self.model.set_train()
        tl = ta = tc = tn = tadv = n = 0
        pbar = tqdm(loader, desc="train", leave=False)
        for batch in pbar:
            if self.model.adv_head is not None:        # DANN ramp 0 -> ADV_LAMBDA
                p = self.global_step / self.total_steps
                self.model.grl_lambda = self.cfg.ADV_LAMBDA * (2.0 / (1.0 + math.exp(-10.0 * p)) - 1.0)
            self.opt.zero_grad(set_to_none=True)
            with _autocast(self.use_amp):
                loss, acc, ch = self._step_loss(batch)
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            nn.utils.clip_grad_norm_([p for p in self.model.parameters() if p.requires_grad], self.cfg.GRAD_CLIP)
            self.scaler.step(self.opt); self.scaler.update(); self.sched.step()
            tl += loss.item(); ta += acc; tc += ch; tn += self._nce; n += 1
            tadv += self._advacc if self._advacc is not None else 0.0
            self.global_step += 1
            pbar.set_postfix(loss=f"{tl/n:.3f}", top1=f"{ta/n:.3f}")
        self.hist['train_loss'].append(tl / n); self.hist['train_top1'].append(ta / n)
        self.hist['train_chance'].append(tc / n); self.hist['train_nce'].append(tn / n)
        if self.model.adv_head is not None:
            self.hist['train_adv_acc'].append(tadv / n)
        return tl / n, ta / n

    @torch.no_grad()
    def validate(self, loader):
        self.model.eval()
        tl = ta = tc = tadv = n = 0
        for batch in tqdm(loader, desc="val", leave=False):
            with _autocast(self.use_amp):
                loss, acc, ch = self._step_loss(batch)
            tl += loss.item(); ta += acc; tc += ch; n += 1
            tadv += self._advacc if self._advacc is not None else 0.0
        self.hist['val_loss'].append(tl / n); self.hist['val_top1'].append(ta / n)
        self.hist['val_chance'].append(tc / n)
        if self.model.adv_head is not None:
            self.hist['val_adv_acc'].append(tadv / n)
        return tl / n, ta / n

    def fit(self, train_loader, val_loader):
        if self.cfg.RESUME and os.path.exists(self.path('latest')):
            self.load(self.path('latest'))
        print(f"\n(top-1 chance level is computed per batch after masking and printed each epoch)\n")
        for ep in range(self.start_epoch, self.cfg.EPOCHS):
            print(f"Epoch {ep+1}/{self.cfg.EPOCHS}   lr(head)={self.sched.get_last_lr()[-1]:.2e}")
            trl, tra = self.train_epoch(train_loader)
            val, vaa = self.validate(val_loader)
            print(f"  train  loss {trl:.4f}  top1 {tra*100:6.2f}%  (chance {self.hist['train_chance'][-1]*100:.2f}%)\n"
                  f"  val    loss {val:.4f}  top1 {vaa*100:6.2f}%  (chance {self.hist['val_chance'][-1]*100:.2f}%)")
            if self.model.adv_head is not None:
                print(f"  centre-adversary acc: train {self.hist['train_adv_acc'][-1]*100:.1f}%  val {self.hist['val_adv_acc'][-1]*100:.1f}%"
                      f"  (chance 25%)  lambda {self.model.grl_lambda:.3f}")
            best = val < self.best_val - 1e-4
            if best:
                self.best_val, self.bad = val, 0
                print("  ✓ new best (val loss)")
            else:
                self.bad += 1
            self.save(ep, best)
            if self.bad >= self.cfg.EARLY_STOP_PATIENCE:
                print(f"⏹ early stop after {self.cfg.EARLY_STOP_PATIENCE} epochs without improvement")
                break
        return self.hist


def plot_history(h, path):
    ep = range(1, len(h['train_loss']) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(ep, h['train_loss'], label='train'); ax[0].plot(ep, h['val_loss'], label='val')
    ax[0].set_title('InfoNCE loss'); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(ep, np.array(h['train_top1']) * 100, label='train'); ax[1].plot(ep, np.array(h['val_top1']) * 100, label='val')
    ax[1].set_title('Top-1 positive retrieval (%)'); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=140); plt.close(fig)


# ====================================================================
# PIPELINE
# ====================================================================
def run(check_only=False, smoke=False, hf=None, index=None):
    make_dirs()
    apply_backbone_defaults(Config)
    if smoke:
        Config.EPOCHS, Config.TRAIN_PAIRS_PER_EPOCH, Config.VAL_PAIRS = (
            Config.SMOKE_EPOCHS, Config.SMOKE_TRAIN, Config.SMOKE_VAL)
        Config.RESUME = False
    set_seed(Config.SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("=" * 70)
    print(f"ADJACENCY CONTRASTIVE | {Config.BACKBONE} @ {get_input_size(Config.BACKBONE)}px | "
          f"pos={Config.POS_MODE} r={Config.NEIGHBOR_RADIUS} | seed={Config.SEED} | {device}")
    print(f"batch {Config.BATCH_SIZE} pairs | tau {Config.TEMPERATURE} | center-pure {Config.CENTER_PURE_BATCHES} | "
          f"neg-mask radius {Config.NEG_MASK_RADIUS} | slides/batch {Config.SLIDES_PER_BATCH} | HED p {Config.HED_P}")
    print(f"tag {Config.TAG} | init {Config.INIT_HUB or 'ImageNet'} | adv lambda {Config.ADV_LAMBDA} | "
          f"centres/batch {Config.CENTERS_PER_BATCH} | radius {Config.NEIGHBOR_RADIUS}")
    print("=" * 70)
    if hf is None:
        hf = load_full_dataset()
        if not check_schema(hf):
            sys.exit(1)
        index = SpatialIndex(hf, Config.GRID_SIZE, cache_path=Config.LATTICE_CACHE or f"{Config.CACHE_DIR}/lattices_g3.pkl")
    if check_only:
        print("✓ checks done"); return
    train_loader, val_loader = create_dataloaders(hf, index, Config)
    model = AdjCLModel(Config.BACKBONE, Config.PRETRAINED, Config.FREEZE_STAGES,
                       Config.PROJ_HIDDEN, Config.PROJ_DIM, init_hub=Config.INIT_HUB,
                       adv_classes=len(Config.TRAIN_CENTERS) if Config.ADV_LAMBDA > 0 else 0,
                       adv_hidden=Config.ADV_HIDDEN).to(device)
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✓ dim={model.feature_dim}, {tr:,}/{sum(p.numel() for p in model.parameters()):,} trainable")
    trainer = Trainer(model, device, Config, steps_per_epoch=len(train_loader))
    hist = trainer.fit(train_loader, val_loader)
    final = trainer.path('final')
    torch.save({'model_state_dict': model.state_dict(), 'backbone': Config.BACKBONE,
                'feature_dim': model.feature_dim, 'history': dict(hist), 'best_val': trainer.best_val}, final)
    plot_history(hist, f"{Config.RESULTS_DIR}/{trainer.stem}_curves.png")
    print(f"\n✓ model → {final}\n✓ best checkpoint → {trainer.path('best')}  (val loss {trainer.best_val:.4f})")
    return model, trainer, hist


def parse_args(argv):
    ap = argparse.ArgumentParser()
    ap.add_argument('--check', action='store_true'); ap.add_argument('--smoke', action='store_true')
    ap.add_argument('--backbone', choices=['resnet18', 'vit_small', 'swin_tiny'])
    ap.add_argument('--pos-mode', choices=['neighbor', 'self', 'mixed'])
    ap.add_argument('--radius', type=int); ap.add_argument('--seed', type=int); ap.add_argument('--epochs', type=int)
    ap.add_argument('--temp', type=float); ap.add_argument('--tag'); ap.add_argument('--no-hed', action='store_true')
    ap.add_argument('--no-center-pure', action='store_true'); ap.add_argument('--no-neg-mask', action='store_true')
    ap.add_argument('--lattice-cache')
    a, _ = ap.parse_known_args(argv)           # parse_known_args: harmless inside a notebook
    for arg, attr in [('backbone', 'BACKBONE'), ('pos_mode', 'POS_MODE'), ('radius', 'NEIGHBOR_RADIUS'),
                      ('seed', 'SEED'), ('epochs', 'EPOCHS'), ('temp', 'TEMPERATURE'), ('tag', 'TAG'),
                      ('lattice_cache', 'LATTICE_CACHE')]:
        if getattr(a, arg) is not None:
            setattr(Config, attr, getattr(a, arg))
    if a.no_hed: Config.HED_P = 0.0
    if a.no_center_pure: Config.CENTER_PURE_BATCHES = False
    if a.no_neg_mask: Config.NEG_MASK_RADIUS = None
    return a


# ====================================================================
# RUN  --  edit this block only
# ====================================================================
import gc

SMOKE = False            # True = 2 short epochs on a small sample (read sec/epoch, then set False)
SEED = 42
DEFAULTS = {k: v for k, v in vars(Config).items() if k.isupper()}      # snapshot: every run starts from these
RUNS = [                 # neighbor = adjacency method, self = SimCLR-style control (same init, same augmentation)
    dict(tag='adjclv4-lunit', backbone='vit_small', mode='neighbor', seed=SEED, overrides=dict(INIT_HUB='hf-hub:1aurent/vit_small_patch16_224.lunit_dino')),
    dict(tag='adjclv4-lunit', backbone='vit_small', mode='self', seed=SEED, overrides=dict(INIT_HUB='hf-hub:1aurent/vit_small_patch16_224.lunit_dino')),
]

Config.BACKBONE = 'vit_small'
run(check_only=True)     # schema check + build/cache the patch index (slow only the first time)

for r in RUNS:
    for k, v in DEFAULTS.items():
        setattr(Config, k, v)
    Config.BACKBONE, Config.TAG, Config.POS_MODE, Config.SEED = r['backbone'], r['tag'], r['mode'], r['seed']
    for k, v in r.get('overrides', {}).items():
        setattr(Config, k, v)
    print("\n>>> RUN", r)
    run(smoke=SMOKE)
    gc.collect(); torch.cuda.empty_cache()

for d in (Config.CHECKPOINT_DIR, Config.RESULTS_DIR):
    print(d, sorted(os.listdir(d)))


ADJACENCY CONTRASTIVE | vit_small @ 224px | pos=neighbor r=1 | seed=42 | cuda
batch 64 pairs | tau 0.1 | center-pure True | neg-mask radius 2 | slides/batch 3 | HED p 0.8
tag adjclv4 | init ImageNet | adv lambda 0.0 | centres/batch 1 | radius 1


README.md:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  470MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  469MB            

...

  train  loss 0.3504  top1  94.76%  (chance 0.79%)
  val    loss 0.0806  top1  99.99%  (chance 0.79%)
Epoch 57/60   lr(head)=3.63e-06


  train  loss 0.3736  top1  94.43%  (chance 0.79%)
  val    loss 0.0688  top1  99.99%  (chance 0.79%)
  ✓ new best (val loss)
Epoch 58/60   lr(head)=2.05e-06


  train  loss 0.3479  top1  94.75%  (chance 0.79%)
  val    loss 0.0921  top1  99.99%  (chance 0.79%)
Epoch 59/60   lr(head)=9.10e-07


  train  loss 0.3632  top1  94.73%  (chance 0.79%)
  val    loss 0.0850  top1  99.99%  (chance 0.79%)
Epoch 60/60   lr(head)=2.28e-07


  train  loss 0.3720  top1  94.47%  (chance 0.79%)
  val    loss 0.0692  top1  99.99%  (chance 0.79%)

✓ model → /kaggle/working/checkpoints/adjclv4-lunit_vit_small_self_s42_final.pt
✓ best checkpoint → /kaggle/working/checkpoints/adjclv4-lunit_vit_small_self_s42_best.pt  (val loss 0.0688)
/kaggle/working/checkpoints ['adjclv4-lunit_vit_small_neighbor_s42_best.pt', 'adjclv4-lunit_vit_small_neighbor_s42_final.pt', 'adjclv4-lunit_vit_small_neighbor_s42_latest.pt', 'adjclv4-lunit_vit_small_self_s42_best.pt', 'adjclv4-lunit_vit_small_self_s42_final.pt', 'adjclv4-lunit_vit_small_self_s42_latest.pt']
/kaggle/working/results ['adjclv4-lunit_vit_small_neighbor_s42_curves.png', 'adjclv4-lunit_vit_small_self_s42_curves.png']
